In [1]:
from pynq import MMIO, Clocks
import time

COMMAND_BASE_ADDR = 0x41200000
STATUS_BASE_ADDR = 0x41210000

GPIO_RANGE = 0x10000
GPIO_DATA_OFFSET = 0x00

In [2]:
print(f"FCLK0: {Clocks.fclk0_mhz} MHz")

FCLK0: 100.0 MHz


In [3]:
from pynq import Clocks

# Display the current FPGA clock frequency
print(f"Previous FCLK0: {Clocks.fclk0_mhz} MHz")

# Configure the FPGA clock to match the hardware design
Clocks.fclk0_mhz = 50.0

# Verify the updated clock frequency
print(f"Current FCLK0: {Clocks.fclk0_mhz} MHz")

Previous FCLK0: 100.0 MHz
Current FCLK0: 50.0 MHz


In [4]:
import math

EXPECTED_CLK_MHZ = 50.0

assert math.isclose(
    Clocks.fclk0_mhz,
    EXPECTED_CLK_MHZ,
    rel_tol=0.001
), "Invalid FPGA clock frequency"

print("Clock configuration: PASS")

Clock configuration: PASS


In [5]:
command_gpio = MMIO(COMMAND_BASE_ADDR, GPIO_RANGE)
status_gpio = MMIO(STATUS_BASE_ADDR, GPIO_RANGE)

print(f"MMIO initialization: PASS")

MMIO initialization: PASS


In [6]:
# Read the initial UART bridge status
status = status_gpio.read(GPIO_DATA_OFFSET)

print(f"Status (hex): 0x{status:08X}")
print(f"Status (bin): {status:032b}")

assert status == 0, "Unexpected initial status"

print("Initial status: PASS")

Status (hex): 0x00000000
Status (bin): 00000000000000000000000000000000
Initial status: PASS


In [7]:
# Prepare the first UART transaction
tx_data = 0xA6
request = 1

command = (request << 8) | tx_data

# Write the command to the AXI GPIO
command_gpio.write(GPIO_DATA_OFFSET, command)

print(f"Command: 0x{command:08X}")
print(f"TX data: 0x{tx_data:02X}")
print(f"Request: {request}")

Command: 0x000001A6
TX data: 0xA6
Request: 1


In [8]:
# Status register masks
ACK_MASK = 1 << 8
ACTIVE_MASK = 1 << 9
TX_DONE_MASK = 1 << 12
RX_DONE_MASK = 1 << 13
FRAMING_ERROR_MASK = 1 << 14

# Wait for transaction completion
timeout = 1.0
deadline = time.monotonic() + timeout

while True:
    status = status_gpio.read(GPIO_DATA_OFFSET)

    ack = (status & ACK_MASK) >> 8
    active = bool(status & ACTIVE_MASK)
    tx_done = bool(status & TX_DONE_MASK)
    rx_done = bool(status & RX_DONE_MASK)
    framing_error = bool(status & FRAMING_ERROR_MASK)

    if ack == request and not active:
        if framing_error:
            raise RuntimeError("UART framing error")

        if tx_done and rx_done:
            break

    if time.monotonic() >= deadline:
        raise TimeoutError(
            f"UART transaction timeout: 0x{status:08X}"
        )

print(f"Final status: 0x{status:08X}")
print("Transaction completed: PASS")

Final status: 0x000031A6
Transaction completed: PASS


In [9]:
# Extract and validate the received byte
rx_data = status & 0xFF

print(f"TX data: 0x{tx_data:02X}")
print(f"RX data: 0x{rx_data:02X}")

assert rx_data == tx_data, "UART loopback data mismatch"
assert status == 0x000031A6, "Unexpected final status"

print("UART loopback: PASS")

TX data: 0xA6
RX data: 0xA6
UART loopback: PASS


In [13]:
# Read the current hardware status
status = status_gpio.read(GPIO_DATA_OFFSET)

# Extract and validate the received byte
rx_data = status & 0xFF

print(f"TX data: 0x{tx_data:02X}")
print(f"RX data: 0x{rx_data:02X}")

assert rx_data == tx_data, "UART loopback data mismatch"
assert status == 0x00003055, "Unexpected final status"

print("Second UART loopback: PASS")

TX data: 0x55
RX data: 0x55
Second UART loopback: PASS


In [14]:
def uart_transfer(data: int, timeout: float = 1.0) -> int:
    # Validate input parameters
    if not 0 <= data <= 0xFF:
        raise ValueError("Data must be an 8-bit value")

    if timeout <= 0:
        raise ValueError("Timeout must be positive")

    # Read the current hardware status
    status = status_gpio.read(GPIO_DATA_OFFSET)

    if status & ACTIVE_MASK:
        raise RuntimeError("UART bridge is busy")

    # Toggle the request bit
    ack = (status & ACK_MASK) >> 8
    request = ack ^ 1

    # Build and send the command
    command = (request << 8) | data
    command_gpio.write(GPIO_DATA_OFFSET, command)

    # Wait for transaction completion
    deadline = time.monotonic() + timeout

    while True:
        status = status_gpio.read(GPIO_DATA_OFFSET)

        ack = (status & ACK_MASK) >> 8
        active = bool(status & ACTIVE_MASK)
        tx_done = bool(status & TX_DONE_MASK)
        rx_done = bool(status & RX_DONE_MASK)
        framing_error = bool(status & FRAMING_ERROR_MASK)

        if ack == request and not active:
            if framing_error:
                raise RuntimeError("UART framing error")

            if tx_done and rx_done:
                return status & 0xFF

        if time.monotonic() >= deadline:
            raise TimeoutError(
                f"UART timeout: status=0x{status:08X}"
            )

In [15]:
# Test representative byte patterns
test_values = [0x00, 0xFF, 0x55, 0xAA, 0xA6]

for tx_data in test_values:
    rx_data = uart_transfer(tx_data)

    assert rx_data == tx_data, (
        f"Mismatch: TX=0x{tx_data:02X}, RX=0x{rx_data:02X}"
    )

    print(f"TX: 0x{tx_data:02X} | RX: 0x{rx_data:02X} | PASS")

print("\nMultiple UART transfers: PASS")

TX: 0x00 | RX: 0x00 | PASS
TX: 0xFF | RX: 0xFF | PASS
TX: 0x55 | RX: 0x55 | PASS
TX: 0xAA | RX: 0xAA | PASS
TX: 0xA6 | RX: 0xA6 | PASS

Multiple UART transfers: PASS


In [16]:
# Test all possible 8-bit values
total_tests = 256
passed_tests = 0
start_time = time.monotonic()

for tx_data in range(total_tests):
    rx_data = uart_transfer(tx_data)

    assert rx_data == tx_data, (
        f"Mismatch: TX=0x{tx_data:02X}, RX=0x{rx_data:02X}"
    )

    passed_tests += 1

elapsed_time = time.monotonic() - start_time

print(f"Passed: {passed_tests}/{total_tests}")
print(f"Failed: {total_tests - passed_tests}")
print(f"Execution time: {elapsed_time:.4f} s")
print("Exhaustive UART byte test: PASS")

Passed: 256/256
Failed: 0
Execution time: 0.0510 s
Exhaustive UART byte test: PASS
